# Прогнозування міцності бетону за допомогою нейронної мережі (PyTorch)

**Задача:** розробити модель глибокого навчання для прогнозування міцності бетону на основі його складових компонентів та віку.

**Датасет:** [Concrete Strength Prediction](https://www.kaggle.com/datasets/mchilamwar/predict-concrete-strength) — 1030 зразків бетону з 8 ознаками та 1 цільовою змінною (міцність у МПа).

## Крок 1. Імпорт бібліотек

In [ ]:
# Імпортуємо необхідні бібліотеки для роботи з даними, моделюванням та візуалізацією
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings('ignore')

RANDOM_STATE = 42
torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

## Крок 2. Завантаження та попередній аналіз даних

In [ ]:
# Завантажуємо набір даних Concrete Strength Prediction з GitHub
url = 'https://raw.githubusercontent.com/baihaqiyazid/dataset/main/concrete_data.csv'
df = pd.read_csv(url)

# Виводимо перші 5 рядків для ознайомлення зі структурою
df.head()

In [ ]:
# Переглядаємо базову інформацію про набір даних
df.info()

In [ ]:
# Обчислюємо описову статистику для кожної змінної
df.describe().round(2)

In [ ]:
# Перевіряємо наявність пропущених значень
df.isnull().sum()

Набір даних містить 1030 зразків та 9 змінних (8 ознак + 1 цільова). Усі змінні числові, пропущених значень немає, категоріальні змінні відсутні — кодування не потрібне.

### Візуалізація даних

In [ ]:
# Будуємо розподіл цільової змінної (міцність бетону)
plt.figure(figsize=(8, 4))
sns.histplot(df['Strength'], bins=30, kde=True, color='steelblue')
plt.xlabel('Міцність бетону (МПа)')
plt.ylabel('Кількість')
plt.title('Розподіл цільової змінної — Strength')
plt.tight_layout()
plt.show()

Розподіл міцності наближений до нормального з невеликим правобічним зсувом. Значення коливаються від ~2 до ~82 МПа.

In [ ]:
# Будуємо розподіл кожної ознаки
features = df.columns[:-1]
fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for i, col in enumerate(features):
    ax = axes[i // 4, i % 4]
    sns.histplot(df[col], bins=30, kde=True, ax=ax, color='steelblue')
    ax.set_title(col)

plt.suptitle('Розподіл вхідних ознак', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Будуємо кореляційну матрицю між усіма змінними
plt.figure(figsize=(10, 8))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', mask=mask, square=True)
plt.title('Кореляційна матриця')
plt.tight_layout()
plt.show()

**Спостереження з кореляційної матриці:**
- **Cement** має найвищу позитивну кореляцію з міцністю (~0.50) — цемент є головним компонентом міцності бетону.
- **Superplasticizer** та **Age** також позитивно корелюють з міцністю.
- **Water** має помітну негативну кореляцію з міцністю — надлишок води послаблює бетон.
- Деякі ознаки (Fly Ash, Blast Furnace Slag) мають слабку кореляцію з цільовою змінною, але можуть впливати нелінійно.

In [ ]:
# Будуємо діаграми розсіювання ключових ознак відносно міцності
key_features = ['Cement', 'Water', 'Age', 'Superplasticizer']
fig, axes = plt.subplots(1, 4, figsize=(18, 4))

for i, col in enumerate(key_features):
    axes[i].scatter(df[col], df['Strength'], alpha=0.4, s=15, color='steelblue')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Strength')
    axes[i].set_title(f'{col} vs Strength')

plt.tight_layout()
plt.show()

## Крок 3. Підготовка даних

In [ ]:
# Розділяємо дані на ознаки (X) та цільову змінну (y)
X = df.drop('Strength', axis=1).values
y = df['Strength'].values

print(f'Ознаки (X): {X.shape}')
print(f'Цільова змінна (y): {y.shape}')

In [ ]:
# Розділяємо дані на навчальний (80%) та тестовий (20%) набори
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

print(f'Навчальна вибірка: X={X_train.shape}, y={y_train.shape}')
print(f'Тестова вибірка:   X={X_test.shape}, y={y_test.shape}')

In [ ]:
# Нормалізуємо вхідні дані за допомогою StandardScaler
# fit_transform лише на train, transform на test — щоб уникнути витоку даних (data leakage)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print('Середнє ознак (train) після масштабування:', X_train.mean(axis=0).round(2))
print('Стд. відхилення ознак (train) після масштабування:', X_train.std(axis=0).round(2))

> **Обґрунтування нормалізації:** ознаки мають дуже різні масштаби (наприклад, Cement ~100–500, Superplasticizer ~0–32). `StandardScaler` приводить усі ознаки до спільного масштабу (середнє ≈ 0, стд. відхилення ≈ 1), що прискорює збіжність градієнтного спуску та покращує стабільність навчання.

### Створення Dataset та DataLoader для батчевої обробки

In [ ]:
# Створюємо клас Dataset для зручного батчевого доступу до даних
class ConcreteDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32).unsqueeze(1)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [ ]:
# Створюємо об'єкти Dataset та DataLoader
BATCH_SIZE = 64

train_dataset = ConcreteDataset(X_train, y_train)
test_dataset = ConcreteDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f'Кількість батчів (train): {len(train_loader)}')
print(f'Кількість батчів (test):  {len(test_loader)}')

## Крок 4. Створення моделі нейронної мережі

In [ ]:
# Створюємо клас нейронної мережі для задачі регресії
class ConcreteNet(nn.Module):
    def __init__(self, in_dim):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(in_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
        )

    def forward(self, x):
        return self.network(x)

**Обґрунтування архітектури:**
- **4 лінійних шари** (8→128→64→32→1) із поступовим зменшенням кількості нейронів — дозволяє мережі послідовно виділяти все більш абстрактні ознаки.
- **ReLU** між шарами — додає нелінійність, необхідну для моделювання складної нелінійної залежності міцності бетону від компонентів.
- **Вихідний шар без активації** — для задачі регресії вихід має бути необмеженим числовим значенням (міцність у МПа).
- Кількість нейронів (128→64→32) обрана з урахуванням невеликого розміру датасету (1030 зразків), щоб зберегти баланс між складністю та ризиком перенавчання.

In [ ]:
# Ініціалізуємо модель
model = ConcreteNet(in_dim=X_train.shape[1])
print(model)
print(f'\nКількість параметрів: {sum(p.numel() for p in model.parameters()):,}')

## Крок 5. Налаштування навчання

In [ ]:
# Визначаємо функцію втрат, оптимізатор та гіперпараметри

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
NUM_EPOCHS = 300

**Обґрунтування вибору:**

- **MSELoss (Mean Squared Error)** — стандартна функція втрат для задач регресії. Вона квадратично штрафує великі помилки, що стимулює модель уникати грубих прогнозів. Диференційовна по всій області значень.
- **Adam** — адаптивний оптимізатор, який поєднує переваги SGD з моментом та RMSprop. Автоматично підлаштовує learning rate для кожного параметра, що робить навчання стабільнішим та швидшим у порівнянні з базовим SGD.
- **lr = 0.001** — загальноприйнятий початковий learning rate для Adam.
- **batch_size = 64** — компроміс між шумом градієнта (малі батчі) та точністю оцінки градієнта (великі батчі). Для датасету з 824 зразками в train дає ~13 батчів на епоху.
- **300 епох** — достатня кількість для збіжності на невеликому датасеті.

## Крок 6. Навчання моделі

In [ ]:
# Навчаємо модель із відстеженням loss на train та test
train_losses = []
test_losses = []

for epoch in range(NUM_EPOCHS):

    # === Тренування ===
    model.train()
    epoch_train_loss = 0.0

    for X_batch, y_batch in train_loader:
        predictions = model(X_batch)
        loss = criterion(predictions, y_batch)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_train_loss += loss.item() * X_batch.size(0)

    epoch_train_loss /= len(train_dataset)
    train_losses.append(epoch_train_loss)

    # === Валідація ===
    model.eval()
    epoch_test_loss = 0.0

    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            predictions = model(X_batch)
            loss = criterion(predictions, y_batch)
            epoch_test_loss += loss.item() * X_batch.size(0)

    epoch_test_loss /= len(test_dataset)
    test_losses.append(epoch_test_loss)

    if (epoch + 1) % 50 == 0:
        print(f'Epoch [{epoch+1:3d}/{NUM_EPOCHS}]  '
              f'Train Loss: {epoch_train_loss:.4f}  '
              f'Test Loss: {epoch_test_loss:.4f}')

## Крок 7. Оцінка моделі

In [ ]:
# Обчислюємо прогнози на тестовому наборі
model.eval()
with torch.no_grad():
    X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
    y_pred = model(X_test_tensor).squeeze().numpy()

# Обчислюємо метрики ефективності
mse = mean_squared_error(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print('=== Метрики на тестовому наборі ===')
print(f'MSE  (середньоквадратична похибка):  {mse:.2f}')
print(f'MAE  (середня абсолютна похибка):    {mae:.2f}')
print(f'R²   (коефіцієнт детермінації):      {r2:.4f}')

**Обґрунтування вибору метрик:**

- **MSE** — основна метрика, узгоджена з функцією втрат. Сильніше штрафує великі помилки, що важливо для задач, де грубі помилки є критичними (як у будівництві).
- **MAE** — інтуїтивно зрозуміла метрика: показує, на скільки МПа в середньому помиляється модель. Менш чутлива до викидів, ніж MSE.
- **R²** — показує частку дисперсії цільової змінної, пояснену моделлю. Значення 1.0 = ідеальний прогноз, 0.0 = модель не краща за середнє значення.

## Крок 8. Аналіз результатів та візуалізація

In [ ]:
# Візуалізуємо криву навчання (функція втрат)
plt.figure(figsize=(8, 4))
plt.plot(train_losses, label='Train Loss', alpha=0.8)
plt.plot(test_losses, label='Test Loss', alpha=0.8)
plt.xlabel('Епоха')
plt.ylabel('MSE Loss')
plt.title('Крива навчання — Train vs Test Loss')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Візуалізуємо фактичні та прогнозовані значення
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter plot: факт vs прогноз
axes[0].scatter(y_test, y_pred, alpha=0.5, s=20, color='steelblue')
min_val = min(y_test.min(), y_pred.min())
max_val = max(y_test.max(), y_pred.max())
axes[0].plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=1.5, label='Ідеальний прогноз')
axes[0].set_xlabel('Фактичне значення (МПа)')
axes[0].set_ylabel('Прогнозоване значення (МПа)')
axes[0].set_title(f'Фактичні vs Прогнозовані значення (R² = {r2:.3f})')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Розподіл похибок (залишків)
residuals = y_test - y_pred
axes[1].hist(residuals, bins=25, color='steelblue', edgecolor='white', alpha=0.8)
axes[1].axvline(x=0, color='red', linestyle='--', linewidth=1.5)
axes[1].set_xlabel('Похибка (МПа)')
axes[1].set_ylabel('Кількість')
axes[1].set_title('Розподіл похибок (залишків)')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Візуалізуємо порівняння фактичних та прогнозованих значень по зразках
indices = np.arange(len(y_test))
sorted_idx = np.argsort(y_test)

plt.figure(figsize=(14, 5))
plt.plot(indices, y_test[sorted_idx], label='Фактичне', alpha=0.8, linewidth=1)
plt.plot(indices, y_pred[sorted_idx], label='Прогнозоване', alpha=0.8, linewidth=1)
plt.xlabel('Зразок (відсортовано за фактичним значенням)')
plt.ylabel('Міцність бетону (МПа)')
plt.title('Фактичні vs Прогнозовані значення міцності бетону')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Аналіз важливості ознак

Для оцінки впливу кожної ознаки на прогноз використаємо метод **permutation importance**: випадково перемішаємо значення однієї ознаки та виміряємо, наскільки це погіршить якість прогнозу.

In [ ]:
# Обчислюємо важливість ознак методом permutation importance
feature_names = df.columns[:-1].tolist()
baseline_mse = mean_squared_error(y_test, y_pred)
importances = []

model.eval()
for i in range(X_test.shape[1]):
    X_test_permuted = X_test.copy()
    np.random.shuffle(X_test_permuted[:, i])

    with torch.no_grad():
        y_pred_perm = model(torch.tensor(X_test_permuted, dtype=torch.float32)).squeeze().numpy()

    perm_mse = mean_squared_error(y_test, y_pred_perm)
    importances.append(perm_mse - baseline_mse)

# Сортуємо та візуалізуємо
importance_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances})
importance_df = importance_df.sort_values('Importance', ascending=True)

plt.figure(figsize=(8, 5))
plt.barh(importance_df['Feature'], importance_df['Importance'], color='steelblue')
plt.xlabel('Збільшення MSE при перемішуванні ознаки')
plt.title('Важливість ознак (Permutation Importance)')
plt.tight_layout()
plt.show()

## Крок 9. Оптимізація моделі

Спробуємо покращити результати, використовуючи:
- Глибшу архітектуру з **Batch Normalization** та **Dropout** для регуляризації
- Планувальник learning rate (**ReduceLROnPlateau**) для адаптивного зменшення кроку навчання

In [ ]:
# Створюємо покращену модель з BatchNorm та Dropout
class ConcreteNetV2(nn.Module):
    def __init__(self, in_dim):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(in_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),

            nn.Linear(64, 32),
            nn.ReLU(),

            nn.Linear(32, 1),
        )

    def forward(self, x):
        return self.network(x)

**Зміни у покращеній моделі:**

- **Batch Normalization** — нормалізує активації між шарами, стабілізує навчання та дозволяє використовувати більший learning rate.
- **Dropout (0.2)** — випадково «вимикає» 20% нейронів під час тренування, зменшуючи ризик перенавчання.
- **Більше нейронів** (256→128→64→32→1) — збільшує ємність моделі для виявлення складніших закономірностей.
- **ReduceLROnPlateau** — автоматично зменшує learning rate, коли валідаційний loss перестає покращуватися.

In [ ]:
# Ініціалізуємо покращену модель з новими гіперпараметрами
torch.manual_seed(RANDOM_STATE)

model_v2 = ConcreteNetV2(in_dim=X_train.shape[1])
criterion_v2 = nn.MSELoss()
optimizer_v2 = torch.optim.Adam(model_v2.parameters(), lr=0.001)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_v2, mode='min', factor=0.5, patience=20
)

NUM_EPOCHS_V2 = 500

print(model_v2)
print(f'\nКількість параметрів: {sum(p.numel() for p in model_v2.parameters()):,}')

In [ ]:
# Навчаємо покращену модель
train_losses_v2 = []
test_losses_v2 = []

for epoch in range(NUM_EPOCHS_V2):

    # === Тренування ===
    model_v2.train()
    epoch_train_loss = 0.0

    for X_batch, y_batch in train_loader:
        predictions = model_v2(X_batch)
        loss = criterion_v2(predictions, y_batch)

        optimizer_v2.zero_grad()
        loss.backward()
        optimizer_v2.step()

        epoch_train_loss += loss.item() * X_batch.size(0)

    epoch_train_loss /= len(train_dataset)
    train_losses_v2.append(epoch_train_loss)

    # === Валідація ===
    model_v2.eval()
    epoch_test_loss = 0.0

    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            predictions = model_v2(X_batch)
            loss = criterion_v2(predictions, y_batch)
            epoch_test_loss += loss.item() * X_batch.size(0)

    epoch_test_loss /= len(test_dataset)
    test_losses_v2.append(epoch_test_loss)

    scheduler.step(epoch_test_loss)

    if (epoch + 1) % 100 == 0:
        current_lr = optimizer_v2.param_groups[0]['lr']
        print(f'Epoch [{epoch+1:3d}/{NUM_EPOCHS_V2}]  '
              f'Train Loss: {epoch_train_loss:.4f}  '
              f'Test Loss: {epoch_test_loss:.4f}  '
              f'LR: {current_lr:.6f}')

In [ ]:
# Обчислюємо метрики покращеної моделі на тестовому наборі
model_v2.eval()
with torch.no_grad():
    X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
    y_pred_v2 = model_v2(X_test_tensor).squeeze().numpy()

mse_v2 = mean_squared_error(y_test, y_pred_v2)
mae_v2 = mean_absolute_error(y_test, y_pred_v2)
r2_v2 = r2_score(y_test, y_pred_v2)

print('=== Метрики покращеної моделі (V2) ===')
print(f'MSE:  {mse_v2:.2f}')
print(f'MAE:  {mae_v2:.2f}')
print(f'R²:   {r2_v2:.4f}')

print('\n=== Порівняння з базовою моделлю ===')
print(f'{"Метрика":<10} {"Базова":>10} {"Покращена":>12} {"Зміна":>10}')
print('-' * 45)
print(f'{"MSE":<10} {mse:>10.2f} {mse_v2:>12.2f} {mse_v2 - mse:>+10.2f}')
print(f'{"MAE":<10} {mae:>10.2f} {mae_v2:>12.2f} {mae_v2 - mae:>+10.2f}')
print(f'{"R²":<10} {r2:>10.4f} {r2_v2:>12.4f} {r2_v2 - r2:>+10.4f}')

In [ ]:
# Візуалізуємо порівняння кривих навчання обох моделей
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Базова модель
axes[0].plot(train_losses, label='Train', alpha=0.8)
axes[0].plot(test_losses, label='Test', alpha=0.8)
axes[0].set_xlabel('Епоха')
axes[0].set_ylabel('MSE Loss')
axes[0].set_title('Базова модель (ConcreteNet)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Покращена модель
axes[1].plot(train_losses_v2, label='Train', alpha=0.8)
axes[1].plot(test_losses_v2, label='Test', alpha=0.8)
axes[1].set_xlabel('Епоха')
axes[1].set_ylabel('MSE Loss')
axes[1].set_title('Покращена модель (ConcreteNetV2)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Візуалізуємо порівняння прогнозів обох моделей
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, y_p, title, r2_val in [
    (axes[0], y_pred, 'Базова модель', r2),
    (axes[1], y_pred_v2, 'Покращена модель', r2_v2),
]:
    ax.scatter(y_test, y_p, alpha=0.5, s=20, color='steelblue')
    min_val = min(y_test.min(), y_p.min())
    max_val = max(y_test.max(), y_p.max())
    ax.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=1.5)
    ax.set_xlabel('Фактичне значення (МПа)')
    ax.set_ylabel('Прогнозоване значення (МПа)')
    ax.set_title(f'{title} (R² = {r2_val:.3f})')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Крок 10. Висновки

### Підсумок результатів навчання

У ході роботи було розроблено дві моделі нейронних мереж для прогнозування міцності бетону:

1. **Базова модель (ConcreteNet)** — 4 лінійних шари з ReLU, навчена за 300 епох.
2. **Покращена модель (ConcreteNetV2)** — 5 лінійних шарів з BatchNorm, Dropout та планувальником learning rate, навчена за 500 епох.

### Аналіз результатів

- Криві навчання обох моделей демонструють стабільне зниження loss без ознак суттєвого перенавчання (train та test loss рухаються синхронно).
- Розподіл залишків (похибок) наближений до нормального та центрований навколо нуля, що свідчить про відсутність систематичного зміщення прогнозів.
- Покращена модель (V2) з BatchNorm та Dropout показує кращу генералізацію завдяки регуляризації.

### Найважливіші фактори міцності бетону

За результатами аналізу permutation importance та кореляційної матриці:

1. **Cement** — головний компонент, що визначає міцність.
2. **Age** — час витримки суттєво впливає на набір міцності.
3. **Water** — надлишок води знижує міцність (негативна кореляція).
4. **Superplasticizer** — пластифікатор покращує міцність за рахунок зменшення потреби у воді.

### Можливі шляхи подальшого покращення

1. **Feature Engineering** — додавання нових ознак, наприклад, відношення вода/цемент (Water-to-Cement ratio), яке є класичним показником у будівельній галузі.
2. **Cross-Validation** — використання k-fold крос-валідації для більш надійної оцінки якості моделі на невеликому датасеті.
3. **Ансамблювання** — об'єднання прогнозів кількох моделей для зменшення дисперсії прогнозів.
4. **Пошук гіперпараметрів** — систематичний підбір learning rate, batch size, кількості шарів та нейронів за допомогою Grid Search (хоч це може бути дорого).
5. **Збільшення даних** — датасет відносно малий (1030 зразків); більший обсяг даних дозволив би навчити складнішу модель без перенавчання.